In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project"

DATA_PATH = f"{BASE_PATH}/data/BreastDCEDL_ISPY1_min_crop"
CLEANED_PATH = f"{BASE_PATH}/data/cleaned"

print("Original data:", DATA_PATH)
print("Cleaned data:", CLEANED_PATH)

In [ ]:
import pandas as pd
import os

master_path = os.path.join(CLEANED_PATH, "patient_master.csv")

patient_df = pd.read_csv(master_path)

print("Patients:", len(patient_df))
print("\nSplit:")
print(patient_df["split"].value_counts())

print("\nColumns:")
print(patient_df.columns.tolist())

In [ ]:
import os

# nnU-Net dataset location
NNUNET_RAW = os.path.join(BASE_PATH, "nnUNet_raw")
DATASET_PATH = os.path.join(NNUNET_RAW, "Dataset001_BreastDCEDL")

# Create required folders
folders = [
    os.path.join(DATASET_PATH, "imagesTr"),
    os.path.join(DATASET_PATH, "labelsTr"),
    os.path.join(DATASET_PATH, "imagesTs")
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Created:")
for folder in folders:
    print(folder)

In [ ]:
!pip install -q nnunetv2

In [ ]:
import nnunetv2
print("nnU-Net v2 installed successfully")

In [ ]:
!nnUNetv2_plan_and_preprocess --help | head -n 5

In [ ]:
# Define nnU-Net directories

NNUNET_RAW = os.path.join(BASE_PATH, "nnUNet_raw")
NNUNET_PREPROCESSED = os.path.join(BASE_PATH, "/content/nnUNet_preprocessed")
NNUNET_RESULTS = os.path.join(BASE_PATH, "nnUNet_results")

# Create directories
os.makedirs(NNUNET_RAW, exist_ok=True)
os.makedirs(NNUNET_PREPROCESSED, exist_ok=True)
os.makedirs(NNUNET_RESULTS, exist_ok=True)

# Tell nnU-Net where these directories are
os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

In [ ]:
import os
import json

NNUNET_DATASET = os.path.join(
    NNUNET_RAW,
    "Dataset001_BreastDCEDL"
)

os.makedirs(NNUNET_DATASET, exist_ok=True)

dataset_json = {
    "channel_names": {
        "0": "pre_contrast",
        "1": "early_post_contrast",
        "2": "late_post_contrast"
    },
    "labels": {
        "background": 0,
        "tumor": 1
    },
    "numTraining": 120,
    "file_ending": ".nii.gz"
}

json_path = os.path.join(NNUNET_DATASET, "dataset.json")

with open(json_path, "w") as f:
    json.dump(dataset_json, f, indent=4)

print("Created:", json_path)
print("\nDataset definition:")
print(json.dumps(dataset_json, indent=4))

In [ ]:
import os
import pandas as pd

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Project paths
BASE_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project"

DATA_PATH = os.path.join(
    BASE_PATH,
    "data",
    "BreastDCEDL_ISPY1_min_crop"
)

CLEANED_PATH = os.path.join(
    BASE_PATH,
    "data",
    "cleaned"
)

# Load patient master table
MASTER_CSV = os.path.join(
    CLEANED_PATH,
    "patient_master.csv"
)

patient_master = pd.read_csv(MASTER_CSV)

print("Patient master loaded successfully.")
print("Rows:", len(patient_master))
print("Columns:", len(patient_master.columns))
print("\nSplit counts:")
print(patient_master["split"].value_counts())

In [ ]:
train_df = patient_master[patient_master["split"] == "train"].copy()

print("Training patients:", len(train_df))
display(train_df[["pid", "dce_0", "dce_1", "dce_2", "mask", "split"]].head(10))

In [ ]:
import os

# nnU-Net paths
NNUNET_RAW = os.path.join(
    BASE_PATH,
    "nnUNet_raw"
)

NNUNET_PREPROCESSED = "/content/nnUNet_preprocessed"

NNUNET_RESULTS = os.path.join(
    BASE_PATH,
    "nnUNet_results"
)

# Set nnU-Net environment variables
os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

print("nnUNet raw:          ", os.environ["nnUNet_raw"])
print("nnUNet preprocessed: ", os.environ["nnUNet_preprocessed"])
print("nnUNet results:      ", os.environ["nnUNet_results"])

In [ ]:
import os

NNUNET_DATASET = os.path.join(
    NNUNET_RAW,
    "Dataset001_BreastDCEDL"
)

print("Dataset folder:")
print(NNUNET_DATASET)

print("\nContents:")
for item in sorted(os.listdir(NNUNET_DATASET)):
    path = os.path.join(NNUNET_DATASET, item)

    if os.path.isdir(path):
        print(f"[DIR]  {item}")
    else:
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"[FILE] {item} ({size_mb:.2f} MB)")

In [ ]:
test_patient = train_df.iloc[0]

print("Test patient:", test_patient["pid"])
print("\nDCE files:")
print("DCE 0:", test_patient["dce_0"])
print("DCE 1:", test_patient["dce_1"])
print("DCE 2:", test_patient["dce_2"])
print("\nMask:")
print(test_patient["mask"])

In [ ]:
import nibabel as nib
import numpy as np

pid = test_patient["pid"]

dce_paths = [
    test_patient["dce_0"],
    test_patient["dce_1"],
    test_patient["dce_2"]
]

mask_path = test_patient["mask"]

print(f"Checking patient: {pid}\n")

# Load DCE volumes
dce_imgs = []
dce_data = []

for i, path in enumerate(dce_paths):
    img = nib.load(path)
    data = img.get_fdata()

    dce_imgs.append(img)
    dce_data.append(data)

    print(f"DCE {i}:")
    print(f"  File exists: {os.path.exists(path)}")
    print(f"  Shape:       {data.shape}")
    print(f"  Dtype:       {data.dtype}")
    print(f"  Spacing:     {img.header.get_zooms()[:3]}")
    print(f"  Min:         {data.min():.2f}")
    print(f"  Max:         {data.max():.2f}")
    print()

# Load mask
mask_img = nib.load(mask_path)
mask_data = mask_img.get_fdata()

print("Mask:")
print(f"  File exists: {os.path.exists(mask_path)}")
print(f"  Shape:       {mask_data.shape}")
print(f"  Unique values: {np.unique(mask_data)}")
print(f"  Tumor voxels:  {np.sum(mask_data == 1)}")
print(f"  Spacing:       {mask_img.header.get_zooms()[:3]}")

# Check consistency
print("\n===== CONSISTENCY CHECKS =====")

print(
    "DCE shapes identical:",
    all(dce_data[0].shape == x.shape for x in dce_data)
)

print(
    "Mask shape matches DCE:",
    mask_data.shape == dce_data[0].shape
)

print(
    "DCE affines identical:",
    all(np.allclose(dce_imgs[0].affine, x.affine) for x in dce_imgs)
)

print(
    "DCE-mask affine matches:",
    np.allclose(dce_imgs[0].affine, mask_img.affine)
)

print(
    "Mask binary:",
    set(np.unique(mask_data)).issubset({0, 1})
)

In [ ]:
import os
import shutil

# nnU-Net training directories
imagesTr = os.path.join(NNUNET_DATASET, "imagesTr")
labelsTr = os.path.join(NNUNET_DATASET, "labelsTr")

os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

# Make sure we are using the 120 training patients
train_df = patient_master[
    patient_master["split"] == "train"
].copy()

print(f"Training patients to copy: {len(train_df)}")

# Copy each patient
for _, row in train_df.iterrows():

    pid = row["pid"]

    # Three DCE channels
    source_dce = [
        row["dce_0"],
        row["dce_1"],
        row["dce_2"]
    ]

    for channel, source_path in enumerate(source_dce):

        destination = os.path.join(
            imagesTr,
            f"{pid}_{channel:04d}.nii.gz"
        )

        shutil.copy2(source_path, destination)

    # Tumor mask
    mask_destination = os.path.join(
        labelsTr,
        f"{pid}.nii.gz"
    )

    shutil.copy2(row["mask"], mask_destination)

print("\nCopy completed.")

In [ ]:
import os
from collections import Counter

# List files
image_files = sorted(
    f for f in os.listdir(imagesTr)
    if f.endswith(".nii.gz")
)

label_files = sorted(
    f for f in os.listdir(labelsTr)
    if f.endswith(".nii.gz")
)

print("===== FILE COUNTS =====")
print("Images:", len(image_files))
print("Labels:", len(label_files))

# Extract patient IDs from image filenames
image_pids = [
    f.rsplit("_", 1)[0]
    for f in image_files
]

label_pids = [
    f.replace(".nii.gz", "")
    for f in label_files
]

image_counts = Counter(image_pids)

print("\n===== CHANNEL COUNTS =====")
print(Counter(image_counts.values()))

print("\n===== EXPECTED =====")
print("Expected images:", 120 * 3)
print("Expected labels:", 120)

# Check every patient has 3 channels
missing_channels = []

for pid in train_df["pid"]:
    expected = {
        f"{pid}_0000.nii.gz",
        f"{pid}_0001.nii.gz",
        f"{pid}_0002.nii.gz"
    }

    actual = {
        f for f in image_files
        if f.startswith(pid + "_")
    }

    if actual != expected:
        missing_channels.append(
            (pid, expected - actual, actual - expected)
        )

# Check masks
missing_labels = [
    pid for pid in train_df["pid"]
    if f"{pid}.nii.gz" not in label_files
]

print("\n===== INTEGRITY =====")
print("Patients with incorrect channels:", len(missing_channels))
print("Patients with missing masks:", len(missing_labels))

if missing_channels:
    print("\nChannel problems:")
    for item in missing_channels[:10]:
        print(item)

if missing_labels:
    print("\nMissing masks:")
    print(missing_labels[:10])

In [ ]:
!nnUNetv2_plan_and_preprocess \
    -d 1 \
    --verify_dataset_integrity \
    --no_pp

In [ ]:
!nnUNetv2_plan_and_preprocess \
    -d 1 \
    --verbose

In [ ]:
# Verify nnU-Net preprocessing output

import os
from pathlib import Path

print("=== nnU-Net Preprocessing Verification ===\n")

# Check the preprocessed dataset folder
preprocessed_dataset = Path(NNUNET_PREPROCESSED) / "Dataset001_BreastDCEDL"

print("Preprocessed dataset:")
print(preprocessed_dataset)
print()

if preprocessed_dataset.exists():
    print("✅ Preprocessed dataset folder exists.")
else:
    print("❌ Preprocessed dataset folder NOT found.")

print("\nContents:")
for item in sorted(preprocessed_dataset.iterdir()):
    if item.is_dir():
        print(f"📁 {item.name}")
    else:
        print(f"📄 {item.name}")

# Check nnU-Net results directory
print("\n=== Results Directory ===")

if os.path.exists(NNUNET_RESULTS):
    print("✅ nnUNet_results exists:")
    print(NNUNET_RESULTS)
else:
    print("❌ nnUNet_results does not exist yet.")

In [ ]:
# Check nnU-Net training options
!nnUNetv2_train --help | head -n 60

In [ ]:
# Check whether any previous nnU-Net training checkpoints exist

from pathlib import Path

results_dataset = Path(NNUNET_RESULTS) / "Dataset001_BreastDCEDL"

print("Training results directory:")
print(results_dataset)
print()

if results_dataset.exists():
    print("Directory already exists.")
    print("\nContents:")
    for item in sorted(results_dataset.iterdir()):
        print("📁" if item.is_dir() else "📄", item.name)
else:
    print("✅ No previous training results found.")
    print("We will start a fresh training run.")

In [ ]:
# Start nnU-Net 3D full-resolution training
# Fold 0 | GPU | Checkpointing enabled

!nnUNetv2_train \
    1 \
    3d_fullres \
    0 \
    -device cuda

In [ ]:
from pathlib import Path

results_dir = Path(
    "/content/drive/MyDrive/Breast_Cancer_AI_Project/nnUNet_results/"
    "Dataset001_BreastDCEDL/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0"
)

print("Fold 0 results directory:")
print(results_dir)
print()

if results_dir.exists():
    print("✅ Fold 0 directory exists.\n")

    for f in sorted(results_dir.iterdir()):
        if f.is_file():
            size_mb = f.stat().st_size / (1024**2)
            print(f"📄 {f.name:<40} {size_mb:.1f} MB")
else:
    print("❌ Fold 0 directory was not found.")

In [ ]:
from pathlib import Path

checkpoint = Path(
    "/content/drive/MyDrive/Breast_Cancer_AI_Project/"
    "nnUNet_results/Dataset001_BreastDCEDL/"
    "nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/"
    "checkpoint_best.pth"
)

print("Checkpoint:", checkpoint)
print("Exists:", checkpoint.exists())

if checkpoint.exists():
    print(f"Size: {checkpoint.stat().st_size / (1024**3):.2f} GB")
    print("✅ Fold 0 best checkpoint is safely stored on Google Drive.")

In [ ]:
import os

preprocessed_dataset = "/content/nnUNet_preprocessed/Dataset001_BreastDCEDL"

print("Preprocessed data exists:", os.path.exists(preprocessed_dataset))

if os.path.exists(preprocessed_dataset):
    print("Contents:")
    print(os.listdir(preprocessed_dataset))
else:
    print("❌ Preprocessed data is gone from this Colab runtime.")

In [ ]:
# Restore nnU-Net environment after the runtime reconnect

import os

BASE_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project"

NNUNET_RAW = os.path.join(BASE_PATH, "nnUNet_raw")
NNUNET_PREPROCESSED = "/content/nnUNet_preprocessed"
NNUNET_RESULTS = os.path.join(BASE_PATH, "nnUNet_results")

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

print("✅ nnU-Net environment restored")
print("Raw:          ", NNUNET_RAW)
print("Preprocessed: ", NNUNET_PREPROCESSED)
print("Results:      ", NNUNET_RESULTS)

In [ ]:
# Train nnU-Net 3D full-resolution — Fold 1
# Checkpointing remains enabled

!nnUNetv2_train \
    1 \
    3d_fullres \
    1 \
    -device cuda

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

fold1_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/nnUNet_results/Dataset001_BreastDCEDL/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_1"

print("Fold 1 exists:", os.path.exists(fold1_path))
print("\nFiles saved in fold_1:")

if os.path.exists(fold1_path):
    for f in sorted(os.listdir(fold1_path)):
        size_mb = os.path.getsize(os.path.join(fold1_path, f)) / (1024**2)
        print(f"{f:45s} {size_mb:,.1f} MB")

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project"

NNUNET_RAW = os.path.join(BASE_PATH, "nnUNet_raw")
NNUNET_PREPROCESSED = os.path.join(BASE_PATH, "nnUNet_preprocessed")
NNUNET_RESULTS = os.path.join(BASE_PATH, "nnUNet_results")

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

print("nnUNet environment restored:")
print("nnUNet_raw          :", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed :", os.environ["nnUNet_preprocessed"])
print("nnUNet_results      :", os.environ["nnUNet_results"])

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("No GPU available.")

In [ ]:
NNUNET_PREPROCESSED = "/content/nnUNet_preprocessed"

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])

In [ ]:
!nnUNetv2_plan_and_preprocess -d 1 --verbose

In [ ]:
import os

preprocessed = "/content/nnUNet_preprocessed/Dataset001_BreastDCEDL"

print("Preprocessed exists:", os.path.exists(preprocessed))

if os.path.exists(preprocessed):
    print("\nContents:")
    for item in os.listdir(preprocessed):
        print(" -", item)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project"

NNUNET_RAW = os.path.join(BASE_PATH, "nnUNet_raw")
NNUNET_PREPROCESSED = "/content/nnUNet_preprocessed"
NNUNET_RESULTS = os.path.join(BASE_PATH, "nnUNet_results")

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

print("Drive:", os.path.exists(BASE_PATH))
print("Raw:", os.path.exists(NNUNET_RAW))
print("Results:", os.path.exists(NNUNET_RESULTS))
print("Preprocessed:", os.path.exists(NNUNET_PREPROCESSED))

In [ ]:
!nnUNetv2_train \
    1 \
    3d_fullres \
    2 \
    -device cuda

In [ ]:
import os

fold2 = "/content/drive/MyDrive/Breast_Cancer_AI_Project/nnUNet_results/Dataset001_BreastDCEDL/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_2"

checkpoint = os.path.join(fold2, "checkpoint_best.pth")

print("Fold 2 folder:", os.path.exists(fold2))
print("Best checkpoint:", os.path.exists(checkpoint))

if os.path.exists(checkpoint):
    size_gb = os.path.getsize(checkpoint) / (1024**3)
    print(f"Checkpoint size: {size_gb:.2f} GB")

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project"

NNUNET_RAW = os.path.join(BASE_PATH, "nnUNet_raw")
NNUNET_PREPROCESSED = "/content/nnUNet_preprocessed"
NNUNET_RESULTS = os.path.join(BASE_PATH, "nnUNet_results")

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNET_PREPROCESSED"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

print("Drive:", os.path.exists(BASE_PATH))
print("Raw:", os.path.exists(NNUNET_RAW))
print("Preprocessed:", os.path.exists(NNUNET_PREPROCESSED))
print("Results:", os.path.exists(NNUNET_RESULTS))

In [ ]:
!nnUNetv2_train 1 3d_fullres 3 -device cuda

In [ ]:
import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project"

NNUNET_RAW = f"{BASE_PATH}/nnUNet_raw"
NNUNET_RESULTS = f"{BASE_PATH}/nnUNet_results"
NNUNET_PREPROCESSED = "/content/nnUNet_preprocessed"

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

In [ ]:
import os

dataset_preprocessed = "/content/nnUNet_preprocessed/Dataset001_BreastDCEDL"

print("Preprocessed folder exists:", os.path.exists(dataset_preprocessed))

if os.path.exists(dataset_preprocessed):
    print("Contents:")
    print(os.listdir(dataset_preprocessed))

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/nnUNet_raw/Dataset001_BreastDCEDL"

print("Dataset exists:", os.path.exists(dataset_path))
print("imagesTr:", len(os.listdir(f"{dataset_path}/imagesTr")))
print("labelsTr:", len(os.listdir(f"{dataset_path}/labelsTr")))
print("dataset.json:", os.path.exists(f"{dataset_path}/dataset.json"))

In [ ]:
!nnUNetv2_plan_and_preprocess -d 1 --verbose

In [ ]:
import os

results_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/nnUNet_results/Dataset001_BreastDCEDL/nnUNetTrainer__nnUNetPlans__3d_fullres"

print("Results folder:", os.path.exists(results_path))

if os.path.exists(results_path):
    print(os.listdir(results_path))

In [ ]:
import json
import os

split_file = "/content/nnUNet_preprocessed/Dataset001_BreastDCEDL/splits_final.json"

print("Split file exists:", os.path.exists(split_file))

if os.path.exists(split_file):
    with open(split_file, "r") as f:
        splits = json.load(f)

    print("Number of folds:", len(splits))

    for i, split in enumerate(splits):
        print(
            f"Fold {i}: "
            f"train={len(split['train'])}, "
            f"val={len(split['val'])}"
        )

In [ ]:
import shutil

src = "/content/nnUNet_preprocessed/Dataset001_BreastDCEDL/splits_final.json"
dst = "/content/drive/MyDrive/Breast_Cancer_AI_Project/nnUNet_raw/Dataset001_BreastDCEDL/splits_final.json"

shutil.copy2(src, dst)

print("Saved split to:", dst)

In [ ]:
!nnUNetv2_train 1 3d_fullres 4 -device cuda

In [ ]:
import os

base = "/content/drive/MyDrive/Breast_Cancer_AI_Project/nnUNet_results/Dataset001_BreastDCEDL/nnUNetTrainer__nnUNetPlans__3d_fullres"

for fold in range(5):
    ckpt = os.path.join(base, f"fold_{fold}", "checkpoint_best.pth")
    print(
        f"Fold {fold}:",
        "✅" if os.path.exists(ckpt) else "❌",
        f"{os.path.getsize(ckpt)/(1024**3):.2f} GB" if os.path.exists(ckpt) else ""
    )

In [ ]:
import os

test_dir = "/content/drive/MyDrive/Breast_Cancer_AI_Project/nnUNet_raw/Dataset001_BreastDCEDL/imagesTs"

test_files = sorted(os.listdir(test_dir))

print("Number of test image files:", len(test_files))
print("First few files:")
print(test_files[:10])

In [ ]:
import pandas as pd

master = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/cleaned/patient_master.csv"

df = pd.read_csv(master)

print(df["split"].value_counts())
print("\nTest patients:")
print(df.loc[df["split"] == "test", "pid"].tolist())

In [ ]:
import os
import shutil
import pandas as pd

BASE_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project"

MASTER_CSV = f"{BASE_PATH}/data/cleaned/patient_master.csv"
ORIGINAL_DCE = f"{BASE_PATH}/data/BreastDCEDL_ISPY1_min_crop/dce"
IMAGES_TS = f"{BASE_PATH}/nnUNet_raw/Dataset001_BreastDCEDL/imagesTs"

df = pd.read_csv(MASTER_CSV)
test_patients = df.loc[df["split"] == "test", "pid"].tolist()

# Make destination fresh
os.makedirs(IMAGES_TS, exist_ok=True)

# Remove anything currently there
for f in os.listdir(IMAGES_TS):
    os.remove(os.path.join(IMAGES_TS, f))

copied = 0
problems = []

all_dce_files = os.listdir(ORIGINAL_DCE)

for pid in test_patients:

    # Get the three DCE files belonging to this patient
    patient_files = sorted([
        f for f in all_dce_files
        if f.startswith(pid + "_") and f.endswith(".nii.gz")
    ])

    if len(patient_files) != 3:
        problems.append((pid, patient_files))
        continue

    # The MinCrop files are the three selected DCE time points
    for channel, src_name in enumerate(patient_files):

        src = os.path.join(ORIGINAL_DCE, src_name)
        dst = os.path.join(IMAGES_TS, f"{pid}_{channel:04d}.nii.gz")

        shutil.copy2(src, dst)
        copied += 1

print("Test patients:", len(test_patients))
print("Files copied:", copied)
print("Problems:", len(problems))

if problems:
    print("\nProblems:")
    for p in problems:
        print(p)

In [ ]:
import pandas as pd

MASTER = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/cleaned/patient_master.csv"

df = pd.read_csv(MASTER)

test_df = df[df["split"] == "test"][["pid", "dce_0", "dce_1", "dce_2"]]

print(test_df.to_string(index=False))

In [ ]:
import os
import shutil
import pandas as pd

BASE = "/content/drive/MyDrive/Breast_Cancer_AI_Project"

MASTER = f"{BASE}/data/cleaned/patient_master.csv"
TEST_DIR = "/content/test_imagesTs"

# Read master table
df = pd.read_csv(MASTER)

test_df = df[df["split"] == "test"].copy()

# Create fresh local test directory
if os.path.exists(TEST_DIR):
    shutil.rmtree(TEST_DIR)

os.makedirs(TEST_DIR)

copied = 0
missing = []

# Use the EXACT dce_0 / dce_1 / dce_2 paths
# recorded in patient_master.csv
for _, row in test_df.iterrows():

    pid = row["pid"]

    for channel in range(3):

        source = row[f"dce_{channel}"]

        if not os.path.exists(source):
            missing.append((pid, channel, source))
            continue

        destination = os.path.join(
            TEST_DIR,
            f"{pid}_{channel:04d}.nii.gz"
        )

        shutil.copy2(source, destination)
        copied += 1

print("Test patients:", len(test_df))
print("Files copied:", copied)
print("Missing:", len(missing))

if missing:
    print("\nMissing:")
    for item in missing:
        print(item)

In [ ]:
files = sorted(os.listdir(TEST_DIR))

print("Local test files:", len(files))

for f in files[:15]:
    print(f)

In [ ]:
import os
import subprocess

TEST_DIR = "/content/test_imagesTs"
BASE_RESULTS = (
    "/content/drive/MyDrive/Breast_Cancer_AI_Project/"
    "nnUNet_results/Dataset001_BreastDCEDL/"
    "nnUNetTrainer__nnUNetPlans__3d_fullres"
)

folds = [0, 1, 2, 3, 4]

for fold in folds:

    checkpoint = os.path.join(
        BASE_RESULTS,
        f"fold_{fold}",
        "checkpoint_best.pth"
    )

    output_dir = f"/content/predictions_fold{fold}"

    print("\n" + "=" * 70)
    print(f"STARTING FOLD {fold}")
    print("=" * 70)

    if not os.path.exists(checkpoint):
        print(f"❌ Checkpoint not found: {checkpoint}")
        continue

    # Fresh output directory for this fold
    os.makedirs(output_dir, exist_ok=True)

    cmd = [
        "nnUNetv2_predict",
        "-d", "1",
        "-i", TEST_DIR,
        "-o", output_dir,
        "-f", str(fold),
        "-c", "3d_fullres",
        "-chk", "checkpoint_best.pth"
    ]

    result = subprocess.run(cmd)

    if result.returncode == 0:
        predictions = [
            f for f in os.listdir(output_dir)
            if f.endswith(".nii.gz")
        ]

        print(f"\n✅ Fold {fold} completed.")
        print(f"Predictions generated: {len(predictions)} / 26")

    else:
        print(f"\n❌ Fold {fold} failed with return code {result.returncode}")

print("\n" + "=" * 70)
print("ALL REQUESTED FOLDS FINISHED")
print("=" * 70)

In [ ]:
import os

for fold in [0, 1, 2, 3, 4]:
    folder = f"/content/predictions_fold{fold}"

    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if f.endswith(".nii.gz")]
        print(f"Fold {fold}: ✅ EXISTS — {len(files)} prediction files")
    else:
        print(f"Fold {fold}: ❌ NOT FOUND")

In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd

# --------------------------------------------------
# PATHS
# --------------------------------------------------

DATA_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop"
MASK_DIR = os.path.join(DATA_PATH, "mask")

folds = [0, 1, 2, 3, 4]

# --------------------------------------------------
# METRIC FUNCTIONS
# --------------------------------------------------

def dice_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    intersection = np.logical_and(pred, gt).sum()

    if pred.sum() + gt.sum() == 0:
        return 1.0

    return (2 * intersection) / (pred.sum() + gt.sum())


def iou_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()

    if union == 0:
        return 1.0

    return intersection / union


# --------------------------------------------------
# EVALUATE ALL FOLDS
# --------------------------------------------------

all_results = []

for fold in folds:

    pred_dir = f"/content/predictions_fold{fold}"

    print("\n" + "=" * 70)
    print(f"EVALUATING FOLD {fold}")
    print("=" * 70)

    if not os.path.exists(pred_dir):
        print(f"❌ Prediction folder not found: {pred_dir}")
        continue

    prediction_files = sorted([
        f for f in os.listdir(pred_dir)
        if f.endswith(".nii.gz")
    ])

    print(f"Predictions found: {len(prediction_files)}")

    fold_results = []

    for filename in prediction_files:

        # Prediction: ISPY1_1029.nii.gz
        pid = filename.replace(".nii.gz", "")

        pred_path = os.path.join(pred_dir, filename)

        # Ground truth:
        # ISPY1_1029_spy1_vis1_mask.nii.gz
        gt_filename = f"{pid}_spy1_vis1_mask.nii.gz"
        gt_path = os.path.join(MASK_DIR, gt_filename)

        if not os.path.exists(gt_path):
            print(f"⚠️ Ground truth missing: {gt_filename}")
            continue

        # Load prediction and ground truth
        pred = nib.load(pred_path).get_fdata() > 0
        gt = nib.load(gt_path).get_fdata() > 0

        # Shape check
        if pred.shape != gt.shape:
            print(
                f"⚠️ Shape mismatch for {pid}: "
                f"pred={pred.shape}, gt={gt.shape}"
            )
            continue

        dice = dice_score(pred, gt)
        iou = iou_score(pred, gt)

        pred_voxels = int(pred.sum())
        gt_voxels = int(gt.sum())

        fold_results.append({
            "fold": fold,
            "pid": pid,
            "dice": dice,
            "iou": iou,
            "gt_voxels": gt_voxels,
            "pred_voxels": pred_voxels
        })

    # Add this fold's results
    all_results.extend(fold_results)

    # --------------------------------------------------
    # FOLD SUMMARY
    # --------------------------------------------------

    if fold_results:

        fold_df = pd.DataFrame(fold_results)

        print("\nFold Results")
        print("-" * 40)
        print(f"Patients evaluated : {len(fold_df)}")
        print(f"Mean Dice          : {fold_df['dice'].mean():.4f}")
        print(f"Median Dice        : {fold_df['dice'].median():.4f}")
        print(f"Mean IoU           : {fold_df['iou'].mean():.4f}")
        print(f"Median IoU         : {fold_df['iou'].median():.4f}")

# --------------------------------------------------
# COMBINED RESULTS
# --------------------------------------------------

results_df = pd.DataFrame(all_results)

print("\n\n" + "=" * 70)
print("FINAL COMPARISON — ALL 5 FOLDS")
print("=" * 70)

if len(results_df) > 0:

    summary = (
        results_df
        .groupby("fold")
        .agg(
            patients=("pid", "count"),
            mean_dice=("dice", "mean"),
            median_dice=("dice", "median"),
            mean_iou=("iou", "mean"),
            median_iou=("iou", "median")
        )
        .reset_index()
    )

    print(summary.to_string(index=False))

    # --------------------------------------------------
    # BEST / WORST PATIENTS
    # --------------------------------------------------

    for fold in folds:

        fold_df = results_df[results_df["fold"] == fold]

        if fold_df.empty:
            continue

        print(f"\n{'=' * 70}")
        print(f"FOLD {fold} — BEST 5")
        print("=" * 70)

        print(
            fold_df
            .sort_values("dice", ascending=False)
            [["pid", "dice", "iou", "gt_voxels", "pred_voxels"]]
            .head(5)
            .to_string(index=False)
        )

        print(f"\nFOLD {fold} — WORST 5")
        print("-" * 70)

        print(
            fold_df
            .sort_values("dice")
            [["pid", "dice", "iou", "gt_voxels", "pred_voxels"]]
            .head(5)
            .to_string(index=False)
        )

else:
    print("❌ No patients were evaluated.")

In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# PATHS
# --------------------------------------------------

DATA_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop"

DCE_DIR = os.path.join(DATA_PATH, "dce")
MASK_DIR = os.path.join(DATA_PATH, "mask")

MASTER_CSV = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/cleaned/patient_master.csv"

master = pd.read_csv(MASTER_CSV)

# --------------------------------------------------
# PATIENTS
# --------------------------------------------------

patients = [
    "ISPY1_1134",   # Good
    "ISPY1_1089",   # Moderate
    "ISPY1_1188",   # Difficult
    "ISPY1_1199"    # Difficult
]

folds = [0, 1, 2, 3, 4]


# --------------------------------------------------
# GET EXACT DCE FILE
# --------------------------------------------------

def get_dce_path(pid, phase=2):

    row = master[master["pid"] == pid]

    if row.empty:
        raise ValueError(f"{pid} not found in patient_master.csv")

    filename = row.iloc[0][f"dce_{phase}"]

    return os.path.join(DCE_DIR, filename)


# --------------------------------------------------
# VISUALIZATION
# --------------------------------------------------

for pid in patients:

    print("\n" + "=" * 70)
    print(pid)
    print("=" * 70)

    # ----------------------------------------------
    # Load MRI
    # ----------------------------------------------

    dce_path = get_dce_path(pid, phase=2)

    image = nib.load(dce_path).get_fdata()

    # ----------------------------------------------
    # Load ground truth
    # ----------------------------------------------

    mask_path = os.path.join(
        MASK_DIR,
        f"{pid}_spy1_vis1_mask.nii.gz"
    )

    gt = nib.load(mask_path).get_fdata() > 0

    # ----------------------------------------------
    # Tumor-containing slices
    # ----------------------------------------------

    tumor_slices = np.where(
        gt.sum(axis=(1, 2)) > 0
    )[0]

    print("Image shape:", image.shape)
    print(
        "Tumor slices:",
        tumor_slices[0],
        "to",
        tumor_slices[-1]
    )
    print(
        "Number of tumor slices:",
        len(tumor_slices)
    )

    # Choose beginning, middle and end
    selected_slices = [
        tumor_slices[0],
        tumor_slices[len(tumor_slices) // 2],
        tumor_slices[-1]
    ]

    # ----------------------------------------------
    # Load predictions
    # ----------------------------------------------

    predictions = {}

    for fold in folds:

        pred_path = (
            f"/content/predictions_fold{fold}/{pid}.nii.gz"
        )

        if os.path.exists(pred_path):

            predictions[fold] = (
                nib.load(pred_path).get_fdata() > 0
            )

    print(
        "Predictions available:",
        list(predictions.keys())
    )

    # ----------------------------------------------
    # 7 COLUMNS:
    #
    # MRI | GT | F0 | F1 | F2 | F3 | F4
    # ----------------------------------------------

    fig, axes = plt.subplots(
        3,
        7,
        figsize=(21, 10)
    )

    for row_idx, slice_idx in enumerate(selected_slices):

        # ------------------------------------------
        # Normalize ONLY for visualization
        # ------------------------------------------

        img = image[slice_idx]

        p_low, p_high = np.percentile(
            img,
            [1, 99]
        )

        img = np.clip(
            (img - p_low) /
            (p_high - p_low + 1e-8),
            0,
            1
        )

        # ------------------------------------------
        # MRI
        # ------------------------------------------

        axes[row_idx, 0].imshow(
            img,
            cmap="gray"
        )

        axes[row_idx, 0].set_title(
            f"MRI\nSlice {slice_idx}"
        )

        # ------------------------------------------
        # Ground Truth
        # ------------------------------------------

        axes[row_idx, 1].imshow(
            img,
            cmap="gray"
        )

        axes[row_idx, 1].contour(
            gt[slice_idx],
            levels=[0.5]
        )

        axes[row_idx, 1].set_title(
            "Ground Truth"
        )

        # ------------------------------------------
        # Fold predictions
        # ------------------------------------------

        for col_idx, fold in enumerate(
            folds,
            start=2
        ):

            axes[row_idx, col_idx].imshow(
                img,
                cmap="gray"
            )

            if fold in predictions:

                axes[row_idx, col_idx].contour(
                    predictions[fold][slice_idx],
                    levels=[0.5]
                )

            axes[row_idx, col_idx].set_title(
                f"Fold {fold}"
            )

        # ------------------------------------------
        # Remove axes
        # ------------------------------------------

        for col in range(7):
            axes[row_idx, col].axis("off")

    fig.suptitle(
        f"{pid} — MRI vs Ground Truth vs 5 Folds",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import os
import subprocess

TEST_DIR = "/content/test_imagesTs"
ENSEMBLE_DIR = "/content/predictions_ensemble"

os.makedirs(ENSEMBLE_DIR, exist_ok=True)

cmd = [
    "nnUNetv2_predict",
    "-d", "1",
    "-i", TEST_DIR,
    "-o", ENSEMBLE_DIR,
    "-f", "0", "1", "2", "3", "4",
    "-c", "3d_fullres",
    "-chk", "checkpoint_best.pth"
]

print("Starting 5-fold ensemble prediction...\n")

result = subprocess.run(cmd)

if result.returncode == 0:
    predictions = [
        f for f in os.listdir(ENSEMBLE_DIR)
        if f.endswith(".nii.gz")
    ]

    print("\n" + "=" * 70)
    print("5-FOLD ENSEMBLE COMPLETE")
    print("=" * 70)
    print(f"Predictions generated: {len(predictions)} / 26")

else:
    print("\n❌ Ensemble prediction failed.")
    print("Return code:", result.returncode)

In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd

# --------------------------------------------------
# PATHS
# --------------------------------------------------

DATA_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop"

MASK_DIR = os.path.join(DATA_PATH, "mask")

ENSEMBLE_DIR = "/content/predictions_ensemble"

# --------------------------------------------------
# METRIC FUNCTIONS
# --------------------------------------------------

def dice_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    intersection = np.logical_and(pred, gt).sum()

    if pred.sum() + gt.sum() == 0:
        return 1.0

    return (2 * intersection) / (pred.sum() + gt.sum())


def iou_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()

    if union == 0:
        return 1.0

    return intersection / union


# --------------------------------------------------
# EVALUATE ENSEMBLE
# --------------------------------------------------

results = []

prediction_files = sorted([
    f for f in os.listdir(ENSEMBLE_DIR)
    if f.endswith(".nii.gz")
])

print("=" * 70)
print("EVALUATING 5-FOLD ENSEMBLE")
print("=" * 70)

print(f"Predictions found: {len(prediction_files)}")

for filename in prediction_files:

    pid = filename.replace(".nii.gz", "")

    pred_path = os.path.join(
        ENSEMBLE_DIR,
        filename
    )

    gt_path = os.path.join(
        MASK_DIR,
        f"{pid}_spy1_vis1_mask.nii.gz"
    )

    if not os.path.exists(gt_path):
        print(f"⚠️ Ground truth missing: {pid}")
        continue

    # Load
    pred = nib.load(pred_path).get_fdata() > 0
    gt = nib.load(gt_path).get_fdata() > 0

    # Shape check
    if pred.shape != gt.shape:
        print(
            f"⚠️ Shape mismatch for {pid}: "
            f"pred={pred.shape}, gt={gt.shape}"
        )
        continue

    # Metrics
    dice = dice_score(pred, gt)
    iou = iou_score(pred, gt)

    pred_voxels = int(pred.sum())
    gt_voxels = int(gt.sum())

    results.append({
        "pid": pid,
        "dice": dice,
        "iou": iou,
        "gt_voxels": gt_voxels,
        "pred_voxels": pred_voxels
    })


# --------------------------------------------------
# RESULTS
# --------------------------------------------------

ensemble_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("5-FOLD ENSEMBLE RESULTS")
print("=" * 70)

print(f"Patients evaluated : {len(ensemble_df)}")
print(f"Mean Dice          : {ensemble_df['dice'].mean():.4f}")
print(f"Median Dice        : {ensemble_df['dice'].median():.4f}")
print(f"Mean IoU           : {ensemble_df['iou'].mean():.4f}")
print(f"Median IoU         : {ensemble_df['iou'].median():.4f}")


# --------------------------------------------------
# BEST / WORST CASES
# --------------------------------------------------

print("\n" + "=" * 70)
print("BEST 5 ENSEMBLE CASES")
print("=" * 70)

print(
    ensemble_df
    .sort_values("dice", ascending=False)
    [["pid", "dice", "iou", "gt_voxels", "pred_voxels"]]
    .head(5)
    .to_string(index=False)
)

print("\n" + "=" * 70)
print("WORST 5 ENSEMBLE CASES")
print("=" * 70)

print(
    ensemble_df
    .sort_values("dice")
    [["pid", "dice", "iou", "gt_voxels", "pred_voxels"]]
    .head(5)
    .to_string(index=False)
)

In [ ]:
import subprocess

result = subprocess.run(
    ["nnUNetv2_predict", "--help"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

In [ ]:
import os
import subprocess

VAL_DIR = "/content/validation_images"
VAL_PROB_DIR = "/content/validation_probabilities"

# Make sure output directory exists
os.makedirs(VAL_PROB_DIR, exist_ok=True)

cmd = [
    "nnUNetv2_predict",
    "-d", "1",
    "-i", VAL_DIR,
    "-o", VAL_PROB_DIR,
    "-f", "0", "1", "2", "3", "4",
    "-c", "3d_fullres",
    "-tr", "nnUNetTrainer",
    "-chk", "checkpoint_best.pth",
    "--save_probabilities",
    "-device", "cuda"
]

print("=" * 70)
print("STARTING 5-FOLD VALIDATION ENSEMBLE")
print("=" * 70)
print("\nCommand:")
print(" ".join(cmd))
print()

result = subprocess.run(cmd)

print("\n" + "=" * 70)

if result.returncode == 0:

    prob_files = sorted([
        f for f in os.listdir(VAL_PROB_DIR)
        if f.endswith(".npz")
    ])

    mask_files = sorted([
        f for f in os.listdir(VAL_PROB_DIR)
        if f.endswith(".nii.gz")
    ])

    print("VALIDATION INFERENCE COMPLETE")
    print("=" * 70)
    print(f"Probability files: {len(prob_files)}")
    print(f"Segmentation files: {len(mask_files)}")

else:

    print("❌ PREDICTION FAILED")
    print("Return code:", result.returncode)

In [ ]:
import os
import numpy as np

prob_dir = "/content/validation_probabilities"

files = sorted([f for f in os.listdir(prob_dir) if f.endswith(".npz")])

print("Number of probability files:", len(files))
print("First file:", files[0])

data = np.load(os.path.join(prob_dir, files[0]))

print("\nContents:")
print(data.files)

for key in data.files:
    print(key, data[key].shape, data[key].dtype)

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd

# --------------------------------------------------
# Paths
# --------------------------------------------------
PROB_DIR = "/content/validation_probabilities"

DATA_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop"
MASK_DIR = os.path.join(DATA_PATH, "mask")

# --------------------------------------------------
# Probability files
# --------------------------------------------------
prob_files = sorted(
    [f for f in os.listdir(PROB_DIR) if f.endswith(".npz")]
)

print("Probability files:", len(prob_files))


# --------------------------------------------------
# Metrics
# --------------------------------------------------
def dice_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    intersection = np.logical_and(pred, gt).sum()

    if pred.sum() == 0 and gt.sum() == 0:
        return 1.0

    return (2.0 * intersection) / (pred.sum() + gt.sum())


def iou_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()

    if union == 0:
        return 1.0

    return intersection / union


# --------------------------------------------------
# Thresholds
# --------------------------------------------------
thresholds = np.arange(0.10, 0.91, 0.05)

results = []


# --------------------------------------------------
# Evaluate
# --------------------------------------------------
for threshold in thresholds:

    dice_scores = []
    iou_scores = []

    for prob_file in prob_files:

        pid = prob_file.replace(".npz", "")

        # Load probabilities
        data = np.load(os.path.join(PROB_DIR, prob_file))

        # Tumor probability
        tumor_prob = data["probabilities"][1]

        # IMPORTANT:
        # nnU-Net probability array is (256, 256, Z)
        # NIfTI GT is (Z, 256, 256)
        tumor_prob = np.transpose(tumor_prob, (2, 0, 1))

        # Threshold
        prediction = tumor_prob >= threshold

        # Ground truth
        mask_file = os.path.join(
            MASK_DIR,
            f"{pid}_spy1_vis1_mask.nii.gz"
        )

        gt = nib.load(mask_file).get_fdata() > 0

        # Safety check
        if prediction.shape != gt.shape:
            raise ValueError(
                f"Shape mismatch for {pid}: "
                f"prediction={prediction.shape}, "
                f"GT={gt.shape}"
            )

        dice_scores.append(
            dice_score(prediction, gt)
        )

        iou_scores.append(
            iou_score(prediction, gt)
        )

    results.append({
        "threshold": round(float(threshold), 2),
        "mean_dice": np.mean(dice_scores),
        "median_dice": np.median(dice_scores),
        "mean_iou": np.mean(iou_scores),
        "median_iou": np.median(iou_scores)
    })


# --------------------------------------------------
# Results
# --------------------------------------------------
results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("VALIDATION THRESHOLD RESULTS")
print("=" * 70)

display(results_df.round(4))


# --------------------------------------------------
# Best threshold
# --------------------------------------------------
best_dice_idx = results_df["mean_dice"].idxmax()
best_iou_idx = results_df["mean_iou"].idxmax()

print("\n" + "=" * 70)
print("BEST THRESHOLDS")
print("=" * 70)

print("\nBest by Mean Dice:")
display(results_df.loc[[best_dice_idx]].round(4))

print("\nBest by Mean IoU:")
display(results_df.loc[[best_iou_idx]].round(4))

In [ ]:
import os
import numpy as np
import nibabel as nib

pid = "ISPY1_1002"

# Probability
prob_file = f"/content/validation_probabilities/{pid}.npz"
data = np.load(prob_file)
prob = data["probabilities"][1]

print("Raw probability shape:", prob.shape)

# nnU-Net segmentation output
seg_file = f"/content/validation_probabilities/{pid}.nii.gz"
seg = nib.load(seg_file).get_fdata()

print("Segmentation shape:", seg.shape)

# Original GT
gt_file = (
    "/content/drive/MyDrive/"
    "Breast_Cancer_AI_Project/data/"
    "BreastDCEDL_ISPY1_min_crop/mask/"
    f"{pid}_spy1_vis1_mask.nii.gz"
)

gt = nib.load(gt_file).get_fdata()

print("GT shape:", gt.shape)

# Print voxel counts
print("\nVoxel counts:")
print("Probability > 0.5:", (prob > 0.5).sum())
print("Segmentation:", (seg > 0).sum())
print("Ground truth:", (gt > 0).sum())

In [ ]:
import numpy as np
import nibabel as nib

pid = "ISPY1_1002"

# -----------------------------
# Load probability
# -----------------------------
prob_data = np.load(
    f"/content/validation_probabilities/{pid}.npz"
)

tumor_prob = prob_data["probabilities"][1]

# Convert probability array to NIfTI array orientation
tumor_prob_t = np.transpose(tumor_prob, (2, 0, 1))

# Threshold
prob_mask = tumor_prob_t >= 0.5


# -----------------------------
# Load nnU-Net segmentation
# -----------------------------
seg = nib.load(
    f"/content/validation_probabilities/{pid}.nii.gz"
).get_fdata() > 0


# -----------------------------
# Compare
# -----------------------------
print("Probability mask shape:", prob_mask.shape)
print("Segmentation shape:", seg.shape)

print("\nProbability voxels:", prob_mask.sum())
print("Segmentation voxels:", seg.sum())

print("\nExact voxel agreement:",
      np.array_equal(prob_mask, seg))

print("Different voxels:",
      np.sum(prob_mask != seg))

In [ ]:
import numpy as np
import nibabel as nib
from itertools import permutations, product

pid = "ISPY1_1002"

# --------------------------------------------------
# Load probability
# --------------------------------------------------
data = np.load(
    f"/content/validation_probabilities/{pid}.npz"
)

prob = data["probabilities"][1]

# --------------------------------------------------
# Load nnU-Net exported segmentation
# --------------------------------------------------
seg = nib.load(
    f"/content/validation_probabilities/{pid}.nii.gz"
).get_fdata() > 0

print("Probability:", prob.shape)
print("Segmentation:", seg.shape)


# --------------------------------------------------
# Try all axis permutations + flips
# --------------------------------------------------
best = None

for perm in permutations(range(3)):

    transformed = np.transpose(prob, perm)

    if transformed.shape != seg.shape:
        continue

    for flips in product([False, True], repeat=3):

        candidate = transformed.copy()

        for axis, do_flip in enumerate(flips):
            if do_flip:
                candidate = np.flip(candidate, axis=axis)

        candidate_mask = candidate >= 0.5

        different = np.sum(candidate_mask != seg)

        if best is None or different < best["different"]:

            best = {
                "permutation": perm,
                "flips": flips,
                "different": int(different),
                "voxels": int(candidate_mask.sum())
            }


print("\n" + "=" * 60)
print("BEST TRANSFORMATION")
print("=" * 60)

print("Permutation:", best["permutation"])
print("Flips:", best["flips"])
print("Different voxels:", best["different"])
print("Candidate voxels:", best["voxels"])
print("Segmentation voxels:", int(seg.sum()))

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd

PROB_DIR = "/content/validation_probabilities"

DATA_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop"
MASK_DIR = os.path.join(DATA_PATH, "mask")

prob_files = sorted(
    f for f in os.listdir(PROB_DIR)
    if f.endswith(".npz")
)

print("Probability files:", len(prob_files))


def dice_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    intersection = np.logical_and(pred, gt).sum()

    if pred.sum() == 0 and gt.sum() == 0:
        return 1.0

    return (2 * intersection) / (pred.sum() + gt.sum())


def iou_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()

    if union == 0:
        return 1.0

    return intersection / union


thresholds = np.arange(0.10, 0.91, 0.05)

results = []

for threshold in thresholds:

    dice_scores = []
    iou_scores = []

    for prob_file in prob_files:

        pid = prob_file.replace(".npz", "")

        # Load tumor probability
        data = np.load(
            os.path.join(PROB_DIR, prob_file)
        )

        tumor_prob = data["probabilities"][1]

        # IMPORTANT:
        # Probability: (256, 256, Z)
        # nnU-Net output / GT: (Z, 256, 256)
        tumor_prob = np.transpose(
            tumor_prob,
            (2, 1, 0)
        )

        # Apply threshold
        prediction = tumor_prob >= threshold

        # Ground truth
        mask_file = os.path.join(
            MASK_DIR,
            f"{pid}_spy1_vis1_mask.nii.gz"
        )

        gt = nib.load(mask_file).get_fdata() > 0

        if prediction.shape != gt.shape:
            raise ValueError(
                f"Shape mismatch for {pid}: "
                f"{prediction.shape} vs {gt.shape}"
            )

        dice_scores.append(
            dice_score(prediction, gt)
        )

        iou_scores.append(
            iou_score(prediction, gt)
        )

    results.append({
        "threshold": round(float(threshold), 2),
        "mean_dice": np.mean(dice_scores),
        "median_dice": np.median(dice_scores),
        "mean_iou": np.mean(iou_scores),
        "median_iou": np.median(iou_scores)
    })


results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("VALIDATION THRESHOLD RESULTS")
print("=" * 70)

display(results_df.round(4))


# Best threshold according to mean Dice
best_idx = results_df["mean_dice"].idxmax()

print("\n" + "=" * 70)
print("BEST THRESHOLD")
print("=" * 70)

display(
    results_df.loc[[best_idx]].round(4)
)

In [ ]:
import os
import numpy as np
import nibabel as nib

PROB_DIR = "/content/validation_probabilities"

dice_scores = []

for f in sorted(os.listdir(PROB_DIR)):

    if not f.endswith(".nii.gz"):
        continue

    pid = f.replace(".nii.gz", "")

    # nnU-Net prediction
    pred = nib.load(
        os.path.join(PROB_DIR, f)
    ).get_fdata() > 0

    # Ground truth
    gt = nib.load(
        f"/content/drive/MyDrive/Breast_Cancer_AI_Project/"
        f"data/BreastDCEDL_ISPY1_min_crop/mask/"
        f"{pid}_spy1_vis1_mask.nii.gz"
    ).get_fdata() > 0

    intersection = np.logical_and(pred, gt).sum()

    dice = (
        2 * intersection /
        (pred.sum() + gt.sum())
    )

    dice_scores.append(dice)

print("Validation patients:", len(dice_scores))
print("Mean Dice:", np.mean(dice_scores))
print("Median Dice:", np.median(dice_scores))

In [ ]:
!rm -rf /content/test_probabilities

!nnUNetv2_predict \
    -d 1 \
    -i /content/test_imagesTs \
    -o /content/test_probabilities \
    -f 0 1 2 3 4 \
    -c 3d_fullres \
    -tr nnUNetTrainer \
    -chk checkpoint_best.pth \
    --save_probabilities \
    -device cuda

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd

# ============================================================
# FROZEN FINAL THRESHOLD
# ============================================================
THRESHOLD = 0.65

PROB_DIR = "/content/test_probabilities"

DATA_PATH = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop"
MASK_DIR = os.path.join(DATA_PATH, "mask")

# ============================================================
# METRICS
# ============================================================
def calculate_metrics(pred, gt):

    pred = pred.astype(bool)
    gt = gt.astype(bool)

    TP = np.logical_and(pred, gt).sum()
    FP = np.logical_and(pred, ~gt).sum()
    FN = np.logical_and(~pred, gt).sum()
    TN = np.logical_and(~pred, ~gt).sum()

    dice = (2 * TP) / (2 * TP + FP + FN) if (2 * TP + FP + FN) > 0 else 1.0

    iou = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 1.0

    precision = TP / (TP + FP) if (TP + FP) > 0 else 1.0

    recall = TP / (TP + FN) if (TP + FN) > 0 else 1.0

    specificity = TN / (TN + FP) if (TN + FP) > 0 else 1.0

    accuracy = (TP + TN) / (TP + TN + FP + FN)

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "accuracy": accuracy,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN
    }


# ============================================================
# FIND PROBABILITY FILES
# ============================================================
prob_files = sorted(
    f for f in os.listdir(PROB_DIR)
    if f.endswith(".npz")
)

print("Probability files:", len(prob_files))
print("Frozen threshold:", THRESHOLD)


# ============================================================
# EVALUATE
# ============================================================
results = []

for prob_file in prob_files:

    pid = prob_file.replace(".npz", "")

    # --------------------------------------------------------
    # Load tumor probability
    # --------------------------------------------------------
    data = np.load(
        os.path.join(PROB_DIR, prob_file)
    )

    tumor_prob = data["probabilities"][1]

    # IMPORTANT:
    # nnU-Net probability array:
    #     (256, 256, Z)
    #
    # GT NIfTI:
    #     (Z, 256, 256)
    #
    # Established exact transformation:
    #     (2, 1, 0)
    # --------------------------------------------------------
    tumor_prob = np.transpose(
        tumor_prob,
        (2, 1, 0)
    )

    # Frozen threshold
    prediction = tumor_prob >= THRESHOLD

    # --------------------------------------------------------
    # Ground truth
    # --------------------------------------------------------
    mask_file = os.path.join(
        MASK_DIR,
        f"{pid}_spy1_vis1_mask.nii.gz"
    )

    gt = nib.load(mask_file).get_fdata() > 0

    # Safety check
    if prediction.shape != gt.shape:
        raise ValueError(
            f"Shape mismatch for {pid}: "
            f"prediction={prediction.shape}, "
            f"GT={gt.shape}"
        )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    metrics = calculate_metrics(prediction, gt)

    results.append({
        "pid": pid,
        **metrics
    })


# ============================================================
# DATAFRAME
# ============================================================
results_df = pd.DataFrame(results)

print("\n" + "=" * 75)
print("FINAL TEST RESULTS — FROZEN THRESHOLD = 0.65")
print("=" * 75)

display(
    results_df[
        [
            "pid",
            "dice",
            "iou",
            "precision",
            "recall",
            "specificity",
            "accuracy"
        ]
    ].round(4)
)


# ============================================================
# OVERALL SUMMARY
# ============================================================
summary = {
    "Mean Dice": results_df["dice"].mean(),
    "Median Dice": results_df["dice"].median(),

    "Mean IoU": results_df["iou"].mean(),
    "Median IoU": results_df["iou"].median(),

    "Mean Precision": results_df["precision"].mean(),
    "Mean Recall": results_df["recall"].mean(),
    "Mean Specificity": results_df["specificity"].mean(),
    "Mean Accuracy": results_df["accuracy"].mean()
}

print("\n" + "=" * 75)
print("OVERALL TEST PERFORMANCE")
print("=" * 75)

for metric, value in summary.items():
    print(f"{metric:25s}: {value:.4f}")